# Outils EN — évaluer un adaptateur (jeu frais + scénarios + juge)

**Runtime : GPU L4.** Secrets : `HF_TOKEN`, **`GEMINI_API_KEY`** (le juge).

Reproduit exactement la mesure de v4 et v5 (`docs/v4_report.md`, `docs/v5_report.md`) :
- **jeu frais 300** : le choix d'outil — v4 et v5 sont à 0,830, c'est la non-régression ;
- **8 scénarios multi-tours** avec outils réels (DuckDuckGo, base de démo), jugés rubrique v2 :
  **c'est là que v5 a échoué** (pertinence 3,62, honnêteté 2,85).

Portes : pertinence ≥ 4,5 · honnêteté ≥ 4 · cohérence ≥ 4,5 · fresh ≥ 0,82.
Le résultat s'affiche entre `===RESULT===` et chaque rapport part sur le Hub.


In [ ]:
# Jetons — le plus propre : Colab > icône clé > secrets HF_TOKEN (écriture), GEMINI_API_KEY, WANDB_API_KEY (optionnel).
import os
from getpass import getpass
try:
    from google.colab import userdata
    read = userdata.get
except Exception:
    read = lambda name: getpass(f"{name} : ")
for name, required in (("HF_TOKEN", True), ("GEMINI_API_KEY", False), ("WANDB_API_KEY", False)):
    try:
        value = read(name)
    except Exception:
        value = "" if not required else getpass(f"{name} : ")
    if value:
        os.environ[name] = value
    elif required:
        raise SystemExit(f"{name} manquant")
print("jetons chargés :", [n for n in ("HF_TOKEN", "GEMINI_API_KEY", "WANDB_API_KEY") if os.environ.get(n)])


## Évaluer (≈ 40 min) — changez `--adapter`/`--tag` pour un autre adaptateur

In [ ]:
import os, subprocess, urllib.request
os.environ.update({
    "LFM2_BRANCH": "rd/pr_rca_eval_baseline",
    "LFM2_JOB": "eval_tc_en",
    "LFM2_ARGS": "--adapter Rcarvalo/lfm25-tc-en-v5_1-adapter --tag v5_1",
    "LFM2_EXTRAS": "serving-liquid,eval,inspect,tooldata",
    "LFM2_ROOT": "/content/repo",
    "LFM2_OUT": "/content/out"
})
urllib.request.urlretrieve("https://raw.githubusercontent.com/rcarvalo/finetuning_s2s_toolcalling/rd/pr_rca_eval_baseline/infra/colab_entrypoint.sh", "/content/entry.sh")
# Au PREMIER PLAN, volontairement : le kernel occupé est ce qui garde la session Colab vivante.
subprocess.run(["bash", "/content/entry.sh"], check=False)


Pour le jeu frais seul (sans clé Gemini) : ajoutez `--skip-scenarios` dans `LFM2_ARGS`.